# Validating a model

This notebook validates the model notebook 07 trained: its false-alarm rate on noise, its efficiency on injections,
its sensitive volume and its ROC. It runs offline on simulated coloured noise.

The validator draws data of its own (from `ValidationConfig.seed`, never the training seed), scores the model's
**logits**, and summarises them, every interval at 1σ (68%) and 2σ (95%) (`interval_sigmas`):

- **FAR** in Hz: background triggers per second of analysed noise, with a Poisson interval counted in the
  background's independent information (a time-slid background reads each detector's data once per slide);
- **efficiency** against SNR at a chosen FAR: the owner's rolling curve (a window of ±0.5 in SNR) with pointwise
  bands, a sigmoid fit with its 50% and 90% SNR points, and the binned values the record keeps;
- **sensitive volume** (Gpc³) and **sensitive distance** (Mpc) within the per-mass reach at each FAR, with three
  intervals;
- **the exact ROC** and its AUC.

An injection is found when its window's score passes the threshold, whatever passes it: the chance that noise alone
passes is subtracted from every efficiency and volume. A target FAR the background is too short to resolve is
reported as unresolved, never as a number.

In [ ]:
import os
os.environ.setdefault("KERAS_BACKEND", "jax")

from pathlib import Path

import numpy as np

import gravyflow as gf
from gravyflow.src.model.examples import (
    DetectionBatches, Gabbard2017, Gabbard2017Config, RunDirectory, TrainingRun, TrainingSchedule,
    materialise,
)

# Runs live outside the package. Set GRAVYFLOW_RUNS to put them elsewhere.
RUN_DIR = Path(os.environ.get("GRAVYFLOW_RUNS", Path.home() / "gravyflow_runs")) / "gabbard_2017_cpu"

from gravyflow.src.model.examples import ValidationDatasetConfig, validation_dataset_args
from gravyflow.src.validate import ValidationConfig, Validator

## The model

`load_model` reads the run's final model and checks its provenance: the file must hold the weights its record names,
and belong to the run's configuration. The model carries the run's record (`run.json`): the split its weights
trained on and the split that selected them.

In [ ]:
run_dir = RunDirectory(RUN_DIR)
model = run_dir.load_model("model", run_dir.run["config_sha256"])
data = run_dir.run["description"]["config"]
print("sample rate:", data["sample_rate_hertz"], "Hz")

## Validation

4096 noise windows (4096 s of background) and 4096 injections at SNR uniform in 0-15. Results are checkpointed to
`validation.h5`, which records the sha256 of the model and of these settings: running the cell again reuses it, and a
different model or setting is refused rather than given stale scores. The validator reads the model's record: the
split it trained on and the split that selected it (`validate`), and refuses to score that split or data split by
another spec. It scores the `test` split by default.

In [ ]:
dataset_args = validation_dataset_args(
    noise_type=gf.NoiseType.COLORED,
    config=ValidationDatasetConfig(sample_rate_hertz=data["sample_rate_hertz"]),
)
validator = Validator.validate(
    model=model,
    name="gabbard_2017_cpu",
    dataset_args=dataset_args,
    config=ValidationConfig(num_examples=4096, batch_size=256),
    checkpoint_file_path=RUN_DIR / "validation" / "validation.h5",
)

## Metrics

`validator.background()` is the noise scores as a `Background`; `threshold_for_far` gives the score threshold for a
target FAR, with the FAR it achieves and its interval. `efficiency_curves` reads the injections at those thresholds:
the rolling curve, the sigmoid fit and the binned record, each with the chance of a noise-only pass subtracted and
intervals carrying the injections', the threshold's and the chance's errors (a joint bootstrap).

This background is simulated noise, so its windows are independent. On real noise of two or more
detectors the validator measures the background by time slides instead: each detector's data slid
against the others' (`ValidationConfig.num_time_slides`, `time_slide_step_seconds`), inside the held-out
split, tiled into windows whose above-threshold neighbours are one trigger (`cluster_seconds`), with
each slide's livetime counted. Real events (`score_real_events`) are read and conditioned exactly as
that background is. They are real data, read under the noise's split, or beside simulated noise like
this the default split, and the validator checks the model against that split before reading any.

In [ ]:
background = validator.background()
print(f"{background.num_windows} windows, {background.livetime_seconds:.0f} s of background")
for far, result in validator.efficiency_curves([1e-1, 1e-2, 1e-3, 1e-4]).items():
    threshold, fit, binned = result.threshold, result.fit, result.binned
    if not threshold.resolved:
        print(f"FAR {far:.0e} Hz: unresolved with {background.livetime_seconds:.0f} s of background")
        continue
    achieved = threshold.achieved
    print(f"FAR {far:.0e} Hz: logit threshold {threshold.threshold:.2f}, achieved {achieved.rate_hz:.1e} Hz "
          f"(1σ interval {achieved.lower_hz:.1e}-{achieved.upper_hz:.1e}); noise alone passes {binned.chance:.1e}")
    if not result.quotable:  # the standard's rule: at least ten independent background events above the threshold
        print(f"    not quoted: {threshold.independent_events} independent background events, fewer than 10")
    print(f"    50% SNR {fit.snr_50:.2f} [2σ {fit.lower[1, 0]:.2f}, {fit.upper[1, 0]:.2f}], "
          f"90% SNR {fit.snr_90:.2f} [2σ {fit.lower[1, 1]:.2f}, {fit.upper[1, 1]:.2f}]")
    for lo, hi, eff, low, up in zip(binned.bin_edges[:-1], binned.bin_edges[1:], binned.efficiency,
                                    binned.lower[1], binned.upper[1]):
        if lo >= 6:
            print(f"    SNR {lo:4.0f}-{hi:<3.0f} efficiency {eff:.2f} [2σ {low:.2f}, {up:.2f}]")
for population, roc in validator.roc_curves().items():
    print(f"ROC {population}: AUC {roc['roc_auc']:.4f}")

## Sensitive volume within the reach

`sensitive_volume_curve` gives the population-averaged volume in which the model finds a source **within the
reach**, at each target FAR, and the sensitive distance, the radius of a sphere of that volume. The reach is per
mass: nothing is findable beyond the distance at which even the best-case source of a mass (its best sky,
orientation and spins) has network SNR about 2 (`volume_reach_snr`), so each injection is placed out to its own
mass's reach. Sources are drawn uniformly in *chirp distance*, d (M_ref / M)^(5/6) with M the chirp mass and M_ref
two neutron stars of 1.4, out to the chirp reach R: a source of chirp mass M lies out to R (M / M_ref)^(5/6), and
stands, when found, for that reach's volume, a weight that keeps V exact. Every number comes with its reach. The
volume is Euclidean (no cosmology) and the masses are in the detector frame, as in MLGWSC-1's `evaluate.py`.

Each volume has three intervals at 1σ and 2σ: the injections' finite number (exact binomial), the background's
finite livetime (the FAR's interval moves the threshold), and both together, by a bootstrap (each replicate
subtracts the chance as the estimator does). The joint interval is the one to quote. Noise alone finds that chance
of the population's whole volume: it is subtracted, and where it exceeds a tenth of what is found the volume rests
on the subtraction: it is quoted with a warning (`chance_dominated`), never refused, since refusing on what is
found would select on the volume's own noise. By the standard's rule for every rate, volume and efficiency, the
points with fewer than ten independent background events at or above the threshold (`background_events`) are not
quoted. `quotable` is the rule.
The dashboard's found fraction against chirp distance, with the chance level drawn on it, shows whether the reach
is long enough: at its far end the fraction should fall to the chance.

In [ ]:
curve = validator.sensitive_volume_curve()
low, high = curve.reach_range_mpc
print(f"Chirp reach R = {curve.reach_mpc:.0f} Mpc: a physical reach of {low:.0f}-{high:.0f} Mpc over the masses")
# The highest and the lowest FAR the background resolves.
for i in np.flatnonzero(curve.resolved)[[0, -1]]:
    print(f"FAR {curve.far_hz[i]:.1e} Hz: V within the reach {curve.volume_within_reach_gpc3[i]:.3g} Gpc^3 "
          f"(joint 2σ interval {curve.joint_lower_gpc3[1, i]:.3g}-{curve.joint_upper_gpc3[1, i]:.3g}), "
          f"distance {curve.distance_within_reach_mpc[i]:.0f} Mpc; noise alone finds {curve.chance_gpc3[i]:.3g} Gpc^3")
print(f"Quotable at {curve.quotable.sum()} of {curve.resolved.sum()} resolved FARs")
# The number a comparison quotes, at the lowest quotable FAR.
if curve.quotable.any():
    q = curve.quote(curve.far_hz[np.flatnonzero(curve.quotable)[-1]])
    print(f"Quoted at {q.far_hz:.1e} Hz: V within the reach (R = {q.reach_mpc:.0f} Mpc chirp distance) "
          f"{q.volume_within_reach_gpc3:.3g} Gpc^3, joint intervals "
          + ", ".join(f"{s:g}σ {lo:.3g}-{hi:.3g}" for s, lo, hi in zip(q.sigmas, q.lower_gpc3, q.upper_gpc3))
          + (" (noise alone finds more than a tenth of it)" if q.chance_dominated else ""))

## The dashboard

`plot` builds an interactive report: efficiency curves with their intervals, sensitive volume and distance
with their three bands, ROC curves, FAR curves with their Poisson bands, the parameter space, and the
worst-scoring examples.

In [ ]:
dashboard = validator.plot(RUN_DIR / "validation" / "validation.html")
dashboard